<a href="https://colab.research.google.com/github/Karthikreddy1010/Deep-Learning_Basic_Projects/blob/main/catsvsdogs_clssification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Cats vs Dogs Classification using Transfer Learning Models
Features: Data Augmentation, ResNet50 & MobileNetV2, Model Comparison
"""

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50, MobileNetV2, EfficientNetB0
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import urllib.request
import zipfile

# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))


# ============================================================================
# STEP 1: DATA PREPARATION & AUGMENTATION
# ============================================================================

def download_cats_dogs_dataset():
    """Download cats vs dogs dataset to current working directory."""
    print("\nSTEP 1: DOWNLOADING CATS VS DOGS DATASET")
    print("="*60)

    current_dir = os.getcwd()
    dataset_dir = os.path.join(current_dir, "cats_dogs_dataset")
    zip_path = os.path.join(current_dir, "cats_and_dogs.zip")
    dataset_url = 'https://download.mlcc.google.com/mledu-datasets/cats_and_dogs_filtered.zip'

    print(f"Working directory: {current_dir}")
    print(f"Dataset will be saved to: {dataset_dir}")

    if os.path.exists(dataset_dir):
        print("Dataset already exists locally!")
        return (
            os.path.join(dataset_dir, "train"),
            os.path.join(dataset_dir, "validation")
        )

    try:
        print(f"Downloading from: {dataset_url}")
        print("This may take a few minutes...")

        def progress_hook(block_num, block_size, total_size):
            downloaded = block_num * block_size
            if total_size > 0:
                percent = min(100, (downloaded * 100) / total_size)
                mb_downloaded = downloaded / (1024 * 1024)
                mb_total = total_size / (1024 * 1024)
                print(f"\rProgress: {percent:.1f}% ({mb_downloaded:.1f}/{mb_total:.1f} MB)",
                      end='', flush=True)

        urllib.request.urlretrieve(dataset_url, zip_path, reporthook=progress_hook)
        print(f"\nDownload completed: {zip_path}")

        print("Extracting dataset...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(current_dir)

        extracted_dir = None
        for item in os.listdir(current_dir):
            if os.path.isdir(item) and "cats_and_dogs" in item.lower():
                extracted_dir = os.path.join(current_dir, item)
                break

        if extracted_dir and extracted_dir != dataset_dir:
            os.rename(extracted_dir, dataset_dir)

        if os.path.exists(zip_path):
            os.remove(zip_path)
            print("Cleaned up zip file")

        print("Dataset setup completed successfully!")
        return (
            os.path.join(dataset_dir, "train"),
            os.path.join(dataset_dir, "validation")
        )

    except Exception as e:
        print(f"Download failed: {str(e)}")
        return None, None


def create_data_augmentation_pipeline():
    """Create data augmentation pipeline for training data."""
    print("\nCreating Data Augmentation Pipeline...")

    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=40,           # Random rotation
        width_shift_range=0.2,       # Random width shift
        height_shift_range=0.2,      # Random height shift
        shear_range=0.2,             # Shear transformation
        zoom_range=0.2,              # Random zoom
        horizontal_flip=True,        # Random horizontal flip
        fill_mode='nearest'          # Fill pixels after transformations
    )

    val_test_datagen = ImageDataGenerator(rescale=1./255)

    return train_datagen, val_test_datagen


def load_data(train_dir, val_dir, img_size=(224, 224), batch_size=32):
    """Load and prepare data using ImageDataGenerator with augmentation."""
    print(f"\nLoading data from directories...")
    print(f"Train dir: {train_dir}")
    print(f"Validation dir: {val_dir}")

    train_datagen, val_datagen = create_data_augmentation_pipeline()

    train_data = train_datagen.flow_from_directory(
        train_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='binary'
    )

    val_data = val_datagen.flow_from_directory(
        val_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='binary',
        shuffle=False
    )

    print(f"Training samples: {train_data.samples}")
    print(f"Validation samples: {val_data.samples}")
    print(f"Classes: {train_data.class_indices}")

    return train_data, val_data


# ============================================================================
# STEP 2: BUILD TRANSFER LEARNING MODELS
# ============================================================================

def build_resnet50_model(img_size=(224, 224)):
    """Build ResNet50-based transfer learning model."""
    print("\nBuilding ResNet50 Model...")

    # Load pre-trained ResNet50
    base_model = ResNet50(
        weights='imagenet',
        input_shape=(*img_size, 3),
        include_top=False
    )

    # Freeze base model weights
    base_model.trainable = False

    # Build new model
    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])

    return model, base_model


def build_mobilenetv2_model(img_size=(224, 224)):
    """Build MobileNetV2-based transfer learning model."""
    print("\nBuilding MobileNetV2 Model...")

    # Load pre-trained MobileNetV2
    base_model = MobileNetV2(
        weights='imagenet',
        input_shape=(*img_size, 3),
        include_top=False
    )

    # Freeze base model weights
    base_model.trainable = False

    # Build new model
    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ])

    return model, base_model


def build_efficientnetb0_model(img_size=(224, 224)):
    """Build EfficientNetB0-based transfer learning model."""
    print("\nBuilding EfficientNetB0 Model...")

    # Load pre-trained EfficientNetB0
    base_model = EfficientNetB0(
        weights='imagenet',
        input_shape=(*img_size, 3),
        include_top=False
    )

    # Freeze base model weights
    base_model.trainable = False

    # Build new model
    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ])

    return model, base_model


# ============================================================================
# STEP 3: TRAINING FUNCTION
# ============================================================================

def train_model(model, train_data, val_data, model_name, epochs=50):
    """Train the model with callbacks."""
    print(f"\n{'='*60}")
    print(f"TRAINING {model_name.upper()}")
    print(f"{'='*60}")

    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    # Callbacks
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        )
    ]

    # Train model
    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )

    return history


def fine_tune_model(model, base_model, train_data, val_data, model_name, epochs=50):
    """Fine-tune the model by unfreezing top layers."""
    print(f"\n{'='*60}")
    print(f"FINE-TUNING {model_name.upper()}")
    print(f"{'='*60}")

    # Unfreeze top layers of base model
    base_model.trainable = True
    for layer in base_model.layers[:-30]:  # Freeze early layers
        layer.trainable = False

    # Recompile with lower learning rate
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
        )
    ]

    # Fine-tune
    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )

    return history


# ============================================================================
# STEP 4: EVALUATION & VISUALIZATION
# ============================================================================

def evaluate_model(model, val_data, model_name):
    """Evaluate model on validation data."""
    print(f"\nEvaluating {model_name}...")

    loss, accuracy = model.evaluate(val_data, verbose=0)

    print(f"{model_name} - Loss: {loss:.4f}, Accuracy: {accuracy:.4f}")

    return loss, accuracy


def plot_training_history(histories, model_names):
    """Plot training history for comparison."""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    for history, name in zip(histories, model_names):
        axes[0].plot(history.history['accuracy'], label=f'{name} Train')
        axes[0].plot(history.history['val_accuracy'], label=f'{name} Val', linestyle='--')

        axes[1].plot(history.history['loss'], label=f'{name} Train')
        axes[1].plot(history.history['val_loss'], label=f'{name} Val', linestyle='--')

    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Model Accuracy Comparison')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Model Loss Comparison')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_comparison.png', dpi=300, bbox_inches='tight')
    print("Training comparison plot saved as 'training_comparison.png'")
    plt.close()


def main():
    """Main training pipeline."""
    print("\n" + "="*60)
    print("CATS VS DOGS CLASSIFICATION WITH TRANSFER LEARNING")
    print("="*60)

    # Step 1: Download dataset
    train_dir, val_dir = download_cats_dogs_dataset()

    if train_dir is None or val_dir is None:
        print("Failed to download dataset. Exiting.")
        return

    # Step 2: Load data with augmentation
    train_data, val_data = load_data(train_dir, val_dir, img_size=(224, 224), batch_size=32)

    # Step 3: Build and train models
    models = {}
    histories = []
    model_names = ['ResNet50', 'MobileNetV2', 'EfficientNetB0']

    # ResNet50
    resnet_model, resnet_base = build_resnet50_model()
    history_resnet = train_model(resnet_model, train_data, val_data, 'ResNet50', epochs=50)
    history_resnet_ft = fine_tune_model(resnet_model, resnet_base, train_data, val_data, 'ResNet50', epochs=50)
    models['ResNet50'] = resnet_model
    histories.append(history_resnet_ft)

    # Reset data generators
    train_data, val_data = load_data(train_dir, val_dir, img_size=(224, 224), batch_size=32)

    # MobileNetV2
    mobilenet_model, mobilenet_base = build_mobilenetv2_model()
    history_mobilenet = train_model(mobilenet_model, train_data, val_data, 'MobileNetV2', epochs=50)
    history_mobilenet_ft = fine_tune_model(mobilenet_model, mobilenet_base, train_data, val_data, 'MobileNetV2', epochs=50)
    models['MobileNetV2'] = mobilenet_model
    histories.append(history_mobilenet_ft)

    # Reset data generators
    train_data, val_data = load_data(train_dir, val_dir, img_size=(224, 224), batch_size=32)

    # EfficientNetB0
    efficientnet_model, efficientnet_base = build_efficientnetb0_model()
    history_efficientnet = train_model(efficientnet_model, train_data, val_data, 'EfficientNetB0', epochs=50)
    history_efficientnet_ft = fine_tune_model(efficientnet_model, efficientnet_base, train_data, val_data, 'EfficientNetB0', epochs=50)
    models['EfficientNetB0'] = efficientnet_model
    histories.append(history_efficientnet_ft)

    # Step 4: Evaluate and compare
    print("\n" + "="*60)
    print("MODEL COMPARISON")
    print("="*60)

    train_data, val_data = load_data(train_dir, val_dir, img_size=(224, 224), batch_size=32)

    results = {}
    for model_name, model in models.items():
        loss, accuracy = evaluate_model(model, val_data, model_name)
        results[model_name] = {'loss': loss, 'accuracy': accuracy}
        model.save(f'{model_name.lower()}_model.keras')
        print(f"✓ Saved {model_name} model")

    # Plot comparison
    plot_training_history(histories, model_names)

    # Summary
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    for model_name, metrics in results.items():
        print(f"{model_name}: Accuracy={metrics['accuracy']:.4f}, Loss={metrics['loss']:.4f}")

    print("\nTraining completed! Models saved:")
    for model_name in model_names:
        print(f"  - {model_name.lower()}_model.keras")


if __name__ == "__main__":
    main()

TensorFlow version: 2.19.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

CATS VS DOGS CLASSIFICATION WITH TRANSFER LEARNING

STEP 1: DOWNLOADING CATS VS DOGS DATASET
Working directory: /content
Dataset will be saved to: /content/cats_dogs_dataset
This may take a few minutes...
Progress: 100.0% (65.4/65.4 MB)
Download completed: /content/cats_and_dogs.zip
Extracting dataset...
Cleaned up zip file
Dataset setup completed successfully!

Loading data from directories...
Train dir: /content/cats_dogs_dataset/train
Validation dir: /content/cats_dogs_dataset/validation

Creating Data Augmentation Pipeline...
Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.
Training samples: 2000
Validation samples: 1000
Classes: {'cats': 0, 'dogs': 1}

Building ResNet50 Model...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step

TRAINING RESNET50
Epoch 1/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 57s 671ms/step - accuracy: 0.5210 - loss: 0.7652 - val

KeyboardInterrupt: 

In [ ]:
"""
Cats vs Dogs Classification Prediction Script
==============================================

Features:
- Supports multiple image formats
- Automatic image preprocessing
- Confidence scoring
- Batch processing
- CSV export

Usage:
    python test.py <input_directory>

Example:
    python test.py ./test_images
"""

import sys
import os
import pandas as pd
import numpy as np
from tensorflow import keras
from tensorflow.keras.models import load_model as keras_load_model
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Configuration
MODEL_PREFERENCE = ['resnet50', 'mobilenetv2', 'efficientnetb0']
OUTPUT_CSV = 'predictions.csv'
VALID_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif', '.webp')
IMG_SIZE = (224, 224)


def load_model(model_path):
    """Load the pre-trained model with error handling."""
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")

    try:
        model = keras_load_model(model_path, safe_mode=False)
        print(f"✓ Model loaded successfully: {model_path}")
        print(f"  Architecture: {model.name}")
        print(f"  Total parameters: {model.count_params():,}")
        return model
    except Exception as e:
        raise RuntimeError(f"Failed to load model: {e}")


def find_model(search_dirs=None):
    """
    Search for a saved model file (.keras or .h5) in common locations.
    Returns model_path or raises FileNotFoundError.
    """
    if search_dirs is None:
        search_dirs = [
            ".",
            os.path.dirname(os.path.abspath(__file__)),
            "/content",
        ]

    for model_name in MODEL_PREFERENCE:
        for directory in search_dirs:
            for ext in [".keras", ".h5", ""]:
                candidate = os.path.join(directory, f"{model_name}_model{ext}")
                if os.path.exists(candidate):
                    return candidate

    # Fallback: find ANY .keras or .h5 in search dirs
    for directory in search_dirs:
        for root, dirs, files in os.walk(directory):
            dirs[:] = [d for d in dirs if not d.startswith('.')]
            for f in files:
                if f.endswith((".keras", ".h5")):
                    return os.path.join(root, f)

    raise FileNotFoundError(
        "No model file found. Please ensure a trained model (e.g. resnet50_model.keras) "
        "exists in the script directory. Run train.py first to generate it."
    )


def preprocess_image(image_path, target_size=IMG_SIZE):
    """
    Load and preprocess image for cats vs dogs prediction.

    Args:
        image_path: Path to image file
        target_size: Target size for resizing (default: 224x224)

    Returns:
        Preprocessed image array ready for prediction
    """
    try:
        # Load image and ensure RGB
        img = Image.open(image_path)
        if img.mode != 'RGB':
            img = img.convert('RGB')

        # Resize to target size
        img = img.resize(target_size, Image.Resampling.LANCZOS)

        # Convert to numpy array and normalize to [0, 1]
        img_array = np.array(img, dtype=np.float32) / 255.0

        # Add batch dimension: (1, H, W, 3)
        img_array = np.expand_dims(img_array, axis=0)

        return img_array

    except Exception as e:
        raise ValueError(f"Failed to preprocess image {image_path}: {e}")


def predict_class(model, img_array, threshold=0.5):
    """
    Predict cat/dog class and confidence.

    Args:
        model: Trained Keras model
        img_array: Preprocessed image array
        threshold: Decision threshold for dog class (default: 0.5)

    Returns:
        Tuple of (predicted_label, predicted_class, confidence_score)
        predicted_class: 0 = Cat, 1 = Dog
    """
    prob = float(model.predict(img_array, verbose=0)[0][0])
    predicted_class = 1 if prob >= threshold else 0
    confidence = prob * 100 if predicted_class == 1 else (1 - prob) * 100
    label = "DOG" if predicted_class == 1 else "CAT"

    return label, predicted_class, round(confidence, 2)


def process_batch(image_files, input_dir, model, batch_size=32):
    """
    Process images in batches for efficiency.

    Args:
        image_files: List of image filenames
        input_dir: Input directory path
        model: Trained model
        batch_size: Number of images to process at once

    Returns:
        List of prediction dictionaries
    """
    predictions = []

    for i in range(0, len(image_files), batch_size):
        batch = image_files[i:i + batch_size]

        for image_file in batch:
            try:
                image_path = os.path.join(input_dir, image_file)
                img_array = preprocess_image(image_path)
                label, pred_class, confidence = predict_class(model, img_array)

                predictions.append({
                    'filename': image_file,
                    'prediction': pred_class,
                    'label': label,
                    'confidence': confidence
                })

                print(f"  ✓ {image_file:40s} → {label} ({pred_class})  "
                      f"(Confidence: {confidence:6.2f}%)")

            except Exception as e:
                print(f"  ✗ {image_file:40s} → ERROR: {str(e)[:50]}")
                predictions.append({
                    'filename': image_file,
                    'prediction': -1,
                    'label': 'ERROR',
                    'confidence': 0.0
                })

    return predictions


def validate_directory(input_dir):
    """Validate input directory and check for images."""
    if not os.path.isdir(input_dir):
        raise NotADirectoryError(f"Input directory not found: {input_dir}")

    image_files = [f for f in os.listdir(input_dir)
                   if f.lower().endswith(VALID_EXTENSIONS)]

    if not image_files:
        raise FileNotFoundError(
            f"No image files found in '{input_dir}'. "
            f"Supported: {', '.join(VALID_EXTENSIONS)}"
        )

    return sorted(image_files)


def process_images(input_dir, model_path=None):
    """
    Main processing function: Load images, predict, and save results.

    Args:
        input_dir: Directory containing test images
        model_path: Path to trained model file (auto-detected if None)

    Returns:
        pandas DataFrame with predictions
    """
    print(f"\n{'='*80}")
    print(f"Processing images from: {input_dir}")
    print(f"{'='*80}")

    # Validate directory
    try:
        image_files = validate_directory(input_dir)
    except Exception as e:
        print(f"\n❌ Error: {e}")
        return None

    # Load model
    try:
        if model_path is None or not os.path.exists(model_path):
            model_path = find_model()
        model = load_model(model_path)
    except Exception as e:
        print(f"\n❌ Error loading model: {e}")
        return None

    # Process images
    print(f"\n🔍 Found {len(image_files)} images. Processing...\n")

    try:
        predictions = process_batch(image_files, input_dir, model)
    except Exception as e:
        print(f"\n❌ Error during processing: {e}")
        return None

    # Create DataFrame
    df = pd.DataFrame(predictions)

    # Statistics
    print(f"\n{'='*80}")
    successful = df[df['prediction'] != -1]
    failed = df[df['prediction'] == -1]
    n_cats = (successful['prediction'] == 0).sum()
    n_dogs = (successful['prediction'] == 1).sum()

    print(f"📊 Processing Summary:")
    print(f"  Total images:       {len(df)}")
    print(f"  Successful:         {len(successful)}")
    print(f"  Failed:             {len(failed)}")

    if len(successful) > 0:
        print(f"\n  CAT predictions:    {n_cats}")
        print(f"  DOG predictions:    {n_dogs}")
        print(f"\n  Average confidence: {successful['confidence'].mean():.2f}%")
        print(f"  Min confidence:     {successful['confidence'].min():.2f}%")
        print(f"  Max confidence:     {successful['confidence'].max():.2f}%")

    # Save predictions
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ Predictions saved to: {OUTPUT_CSV}")
    print(f"{'='*80}")

    return df


def main():
    """Main function for command-line execution."""
    if len(sys.argv) < 2:
        print(f"Usage: python {sys.argv[0]} <input_directory> [model_path]")
        print(f"\nExample: python {sys.argv[0]} ./test_images")
        print(f"          python {sys.argv[0]} ./test_images resnet50_model.keras")
        return 1

    input_dir = sys.argv[1]
    model_path = sys.argv[2] if len(sys.argv) > 2 else None

    try:
        process_images(input_dir, model_path)
        return 0
    except Exception as e:
        print(f"\n❌ Fatal error: {e}")
        return 1


if __name__ == "__main__":
    sys.exit(main())

Only MobileNet V2 code of both training and testing

In [ ]:
"""
Cats vs Dogs Classification using Transfer Learning (MobileNetV2)
Features: Data Augmentation, Fine-Tuning, Model Evaluation
"""

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import urllib.request
import zipfile

# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))


# ============================================================================
# STEP 1: DATA PREPARATION & AUGMENTATION
# ============================================================================

def download_cats_dogs_dataset():
    """Download cats vs dogs dataset to current working directory."""
    print("\nSTEP 1: DOWNLOADING CATS VS DOGS DATASET")
    print("="*60)

    current_dir = os.getcwd()
    dataset_dir = os.path.join(current_dir, "cats_dogs_dataset")
    zip_path = os.path.join(current_dir, "cats_and_dogs.zip")
    dataset_url = 'https://download.mlcc.google.com/mledu-datasets/cats_and_dogs_filtered.zip'

    print(f"Working directory: {current_dir}")
    print(f"Dataset will be saved to: {dataset_dir}")

    if os.path.exists(dataset_dir):
        print("Dataset already exists locally!")
        return (
            os.path.join(dataset_dir, "train"),
            os.path.join(dataset_dir, "validation")
        )

    try:
        print(f"Downloading from: {dataset_url}")
        print("This may take a few minutes...")

        def progress_hook(block_num, block_size, total_size):
            downloaded = block_num * block_size
            if total_size > 0:
                percent = min(100, (downloaded * 100) / total_size)
                mb_downloaded = downloaded / (1024 * 1024)
                mb_total = total_size / (1024 * 1024)
                print(f"\rProgress: {percent:.1f}% ({mb_downloaded:.1f}/{mb_total:.1f} MB)",
                      end='', flush=True)

        urllib.request.urlretrieve(dataset_url, zip_path, reporthook=progress_hook)
        print(f"\nDownload completed: {zip_path}")

        print("Extracting dataset...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(current_dir)

        extracted_dir = None
        for item in os.listdir(current_dir):
            if os.path.isdir(item) and "cats_and_dogs" in item.lower():
                extracted_dir = os.path.join(current_dir, item)
                break

        if extracted_dir and extracted_dir != dataset_dir:
            os.rename(extracted_dir, dataset_dir)

        if os.path.exists(zip_path):
            os.remove(zip_path)
            print("Cleaned up zip file")

        print("Dataset setup completed successfully!")
        return (
            os.path.join(dataset_dir, "train"),
            os.path.join(dataset_dir, "validation")
        )

    except Exception as e:
        print(f"Download failed: {str(e)}")
        return None, None


def create_data_augmentation_pipeline():
    """Create data augmentation pipeline for training data."""
    print("\nCreating Data Augmentation Pipeline...")

    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=40,           # Random rotation
        width_shift_range=0.2,       # Random width shift
        height_shift_range=0.2,      # Random height shift
        shear_range=0.2,             # Shear transformation
        zoom_range=0.2,              # Random zoom
        horizontal_flip=True,        # Random horizontal flip
        fill_mode='nearest'          # Fill pixels after transformations
    )

    val_datagen = ImageDataGenerator(rescale=1./255)

    return train_datagen, val_datagen


def load_data(train_dir, val_dir, img_size=(224, 224), batch_size=32):
    """Load and prepare data using ImageDataGenerator with augmentation."""
    print(f"\nLoading data from directories...")
    print(f"Train dir: {train_dir}")
    print(f"Validation dir: {val_dir}")

    train_datagen, val_datagen = create_data_augmentation_pipeline()

    train_data = train_datagen.flow_from_directory(
        train_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='binary'
    )

    val_data = val_datagen.flow_from_directory(
        val_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='binary',
        shuffle=False
    )

    print(f"Training samples: {train_data.samples}")
    print(f"Validation samples: {val_data.samples}")
    print(f"Classes: {train_data.class_indices}")

    return train_data, val_data


# ============================================================================
# STEP 2: BUILD MOBILENETV2 MODEL
# ============================================================================

def build_mobilenetv2_model(img_size=(224, 224)):
    """Build MobileNetV2-based transfer learning model."""
    print("\nBuilding MobileNetV2 Model...")

    # Load pre-trained MobileNetV2
    base_model = MobileNetV2(
        weights='imagenet',
        input_shape=(*img_size, 3),
        include_top=False
    )

    # Freeze base model weights
    base_model.trainable = False

    # Build new classification head
    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ])

    return model, base_model


# ============================================================================
# STEP 3: TRAINING & FINE-TUNING
# ============================================================================

def train_model(model, train_data, val_data, model_name, epochs=50):
    """Train the model with callbacks."""
    print(f"\n{'='*60}")
    print(f"TRAINING {model_name.upper()}")
    print(f"{'='*60}")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        )
    ]

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )

    return history


def fine_tune_model(model, base_model, train_data, val_data, model_name, epochs=50):
    """Fine-tune the model by unfreezing top layers."""
    print(f"\n{'='*60}")
    print(f"FINE-TUNING {model_name.upper()}")
    print(f"{'='*60}")

    # Unfreeze top layers of base model
    base_model.trainable = True
    for layer in base_model.layers[:-30]:   # Keep early layers frozen
        layer.trainable = False

    # Recompile with lower learning rate
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
        )
    ]

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )

    return history


# ============================================================================
# STEP 4: EVALUATION & VISUALIZATION
# ============================================================================

def evaluate_model(model, val_data, model_name):
    """Evaluate model on validation data."""
    print(f"\nEvaluating {model_name}...")

    loss, accuracy = model.evaluate(val_data, verbose=0)
    print(f"{model_name} - Loss: {loss:.4f}, Accuracy: {accuracy:.4f}")

    return loss, accuracy


def plot_training_history(history, model_name):
    """Plot training and validation accuracy/loss curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Validation', linestyle='--')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title(f'{model_name} — Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Validation', linestyle='--')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title(f'{model_name} — Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{model_name.lower()}_training_history.png', dpi=300, bbox_inches='tight')
    print(f"Training history plot saved as '{model_name.lower()}_training_history.png'")
    plt.close()


# ============================================================================
# MAIN
# ============================================================================

def main():
    """Main training pipeline."""
    print("\n" + "="*60)
    print("CATS VS DOGS CLASSIFICATION — MobileNetV2")
    print("="*60)

    # Step 1: Download dataset
    train_dir, val_dir = download_cats_dogs_dataset()

    if train_dir is None or val_dir is None:
        print("Failed to download dataset. Exiting.")
        return

    # Step 2: Load data with augmentation
    train_data, val_data = load_data(train_dir, val_dir, img_size=(224, 224), batch_size=32)

    # Step 3: Build model
    model, base_model = build_mobilenetv2_model()
    model.summary()

    # Step 4: Initial training (frozen base)
    history_train = train_model(model, train_data, val_data, 'MobileNetV2', epochs=50)

    # Step 5: Fine-tuning (unfreeze top layers)
    train_data, val_data = load_data(train_dir, val_dir, img_size=(224, 224), batch_size=32)
    history_ft = fine_tune_model(model, base_model, train_data, val_data, 'MobileNetV2', epochs=50)

    # Step 6: Evaluate
    print("\n" + "="*60)
    print("FINAL EVALUATION")
    print("="*60)

    train_data, val_data = load_data(train_dir, val_dir, img_size=(224, 224), batch_size=32)
    loss, accuracy = evaluate_model(model, val_data, 'MobileNetV2')

    # Step 7: Save model
    model.save('mobilenetv2_model.keras')
    print(f"✓ Model saved as: mobilenetv2_model.keras")

    # Step 8: Plot fine-tuning history
    plot_training_history(history_ft, 'MobileNetV2')

    # Summary
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    print(f"MobileNetV2: Accuracy={accuracy:.4f}, Loss={loss:.4f}")
    print("\nTraining completed! Model saved: mobilenetv2_model.keras")


if __name__ == "__main__":
    main()

TensorFlow version: 2.19.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

CATS VS DOGS CLASSIFICATION — MobileNetV2

STEP 1: DOWNLOADING CATS VS DOGS DATASET
Working directory: /content
Dataset will be saved to: /content/cats_dogs_dataset
Dataset already exists locally!

Loading data from directories...
Train dir: /content/cats_dogs_dataset/train
Validation dir: /content/cats_dogs_dataset/validation

Creating Data Augmentation Pipeline...
Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.
Training samples: 2000
Validation samples: 1000
Classes: {'cats': 0, 'dogs': 1}

Building MobileNetV2 Model...


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,618,945 (9.99 MB)

 Trainable params: 360,961 (1.38 MB)

 Non-trainable params: 2,257,984 (8.61 MB)


TRAINING MOBILENETV2
Epoch 1/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 54s 639ms/step - accuracy: 0.8370 - loss: 0.3829 - val_accuracy: 0.9730 - val_loss: 0.1165 - learning_rate: 1.0000e-04
Epoch 2/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 30s 485ms/step - accuracy: 0.9290 - loss: 0.1833 - val_accuracy: 0.9770 - val_loss: 0.0778 - learning_rate: 1.0000e-04
Epoch 3/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 28s 438ms/step - accuracy: 0.9320 - loss: 0.1654 - val_accuracy: 0.9800 - val_loss: 0.0682 - learning_rate: 1.0000e-04
Epoch 4/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 27s 424ms/step - accuracy: 0.9420 - loss: 0.1419 - val_accuracy: 0.9800 - val_loss: 0.0630 - learning_rate: 1.0000e-04
Epoch 5/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 27s 423ms/step - accuracy: 0.9455 - loss: 0.1362 - val_accuracy: 0.9760 - val_loss: 0.0668 - learning_rate: 1.0000e-04
Epoch 6/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 27s 432ms/step - accuracy: 0.9530 - loss: 0.1239 - val_accuracy: 0.9780 - val_loss: 0.0620 - learning_rate: 1.0000e-04
Epoch 7/50
63/63 ━━━━━━━━━━━━━━━━━━━

In [9]:
import os
import sys
import zipfile
from datetime import datetime
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
import numpy as np
import urllib.request

print("🐱🐶 CATS VS DOGS SUBMISSION CREATOR")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {keras.__version__}")


# ============================================================================
# STEP 1: CREATE test.py
# ============================================================================

print("\n📄 Creating test.py...")

test_py_content = '''import sys
import os
import pandas as pd
import numpy as np
from PIL import Image
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
import warnings
warnings.filterwarnings(\'ignore\')

# Configuration
WEIGHTS_PATH = \'mobilenetv2_weights.weights.h5\'
IMG_SIZE = (224, 224)


def build_model():
    """Rebuild the exact MobileNetV2 architecture used during training."""
    base_model = MobileNetV2(
        weights=None,
        input_shape=(224, 224, 3),
        include_top=False
    )
    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation=\'relu\'),
        layers.Dropout(0.4),
        layers.Dense(128, activation=\'relu\'),
        layers.Dropout(0.2),
        layers.Dense(1, activation=\'sigmoid\')
    ], name="MobileNetV2_CatsDogs")
    return model


def load_model():
    """Build model architecture and load saved weights."""
    model = build_model()
    # Build the model by passing a dummy input
    model.build((None, 224, 224, 3))
    model.load_weights(WEIGHTS_PATH)
    print(f"✓ Model loaded: {WEIGHTS_PATH}")
    print(f"  Architecture: {model.name}")
    print(f"  Total parameters: {model.count_params():,}")
    return model


def load_image(image_path):
    """
    Load and preprocess image for prediction.

    Args:
        image_path (str): Path to the image file

    Returns:
        Preprocessed numpy array of shape (1, 224, 224, 3)
    """
    img = Image.open(image_path)
    if img.mode != \'RGB\':
        img = img.convert(\'RGB\')
    img = img.resize(IMG_SIZE, Image.Resampling.LANCZOS)
    img_array = np.array(img, dtype=np.float32) / 255.0
    img_array = np.expand_dims(img_array, axis=0)  # (1, 224, 224, 3)
    return img_array


def main(input_dir):
    """
    Main inference function for cats vs dogs classification.

    Args:
        input_dir (str): Directory containing test images

    Returns:
        None (saves predictions to predictions.csv)
    """
    # 1. Load your pre-trained model
    model = load_model()

    # 2. Process all images in input_dir
    predictions = []
    for image_file in os.listdir(input_dir):
        if image_file.lower().endswith((\'.png\', \'.jpg\', \'.jpeg\')):
            # Load and preprocess image
            image = load_image(os.path.join(input_dir, image_file))

            # Make binary prediction (0=cat, 1=dog)
            prob = model.predict(image, verbose=0)[0][0]
            pred = 1 if prob >= 0.5 else 0
            predictions.append({
                \'filename\': image_file,
                \'prediction\': int(pred)  # Must be 0 or 1
            })
            label = "DOG" if pred == 1 else "CAT"
            print(f"  {image_file:40s} → {label} ({pred})")

    # 3. Save predictions to CSV
    df = pd.DataFrame(predictions)
    df.to_csv(\'predictions.csv\', index=False)
    print(f"\\n✅ Saved {len(df)} predictions to predictions.csv")


if __name__ == "__main__":
    if len(sys.argv) != 2:
        print("Usage: python test.py <input_directory>")
        sys.exit(1)

    input_dir = sys.argv[1]
    main(input_dir)
'''

with open('test.py', 'w') as f:
    f.write(test_py_content)

print("  ✓ test.py created")


# ============================================================================
# STEP 2: CREATE requirements.txt
# ============================================================================

print("\n📄 Creating requirements.txt...")

requirements_content = """tensorflow>=2.13.0
numpy>=1.24.0
pandas>=2.0.0
Pillow>=10.0.0
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print("  ✓ requirements.txt created")


# ============================================================================
# STEP 3: DOWNLOAD DATASET
# ============================================================================

def download_cats_dogs_dataset():
    """Download cats vs dogs dataset."""
    print("\n" + "="*60)
    print("STEP 3: DOWNLOADING CATS VS DOGS DATASET")
    print("="*60)

    current_dir = os.getcwd()
    dataset_dir = os.path.join(current_dir, "cats_dogs_dataset")
    zip_path = os.path.join(current_dir, "cats_and_dogs.zip")
    dataset_url = 'https://download.mlcc.google.com/mledu-datasets/cats_and_dogs_filtered.zip'

    if os.path.exists(dataset_dir):
        print("  Dataset already exists locally!")
        return (
            os.path.join(dataset_dir, "train"),
            os.path.join(dataset_dir, "validation")
        )

    print(f"  Downloading from: {dataset_url}")
    print("  This may take a few minutes...")

    def progress_hook(block_num, block_size, total_size):
        downloaded = block_num * block_size
        if total_size > 0:
            percent = min(100, (downloaded * 100) / total_size)
            mb_downloaded = downloaded / (1024 * 1024)
            mb_total = total_size / (1024 * 1024)
            print(f"\r  Progress: {percent:.1f}% ({mb_downloaded:.1f}/{mb_total:.1f} MB)",
                  end='', flush=True)

    urllib.request.urlretrieve(dataset_url, zip_path, reporthook=progress_hook)
    print(f"\n  Download completed!")

    print("  Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(current_dir)

    extracted_dir = None
    for item in os.listdir(current_dir):
        if os.path.isdir(item) and "cats_and_dogs" in item.lower():
            extracted_dir = os.path.join(current_dir, item)
            break

    if extracted_dir and extracted_dir != dataset_dir:
        os.rename(extracted_dir, dataset_dir)

    if os.path.exists(zip_path):
        os.remove(zip_path)

    print("  ✓ Dataset ready!")
    return (
        os.path.join(dataset_dir, "train"),
        os.path.join(dataset_dir, "validation")
    )


# ============================================================================
# STEP 4: BUILD MODEL (reusable function — same arch used in test.py)
# ============================================================================

def build_model():
    """Build the MobileNetV2 transfer learning model."""
    base_model = MobileNetV2(
        weights='imagenet',
        input_shape=(224, 224, 3),
        include_top=False
    )
    base_model.trainable = False

    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ], name="MobileNetV2_CatsDogs")

    return model, base_model


# ============================================================================
# STEP 5: TRAIN MODEL
# ============================================================================

print("\n" + "="*60)
print("STEP 5: TRAINING MobileNetV2 MODEL")
print("This will take 15-20 minutes. Training in progress...")
print("="*60)

tf.random.set_seed(42)
np.random.seed(42)

train_dir, val_dir = download_cats_dogs_dataset()

# Data generators
print("\n  Setting up data generators...")

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir, target_size=(224, 224), batch_size=32, class_mode='binary'
)
val_data = val_datagen.flow_from_directory(
    val_dir, target_size=(224, 224), batch_size=32, class_mode='binary', shuffle=False
)

print(f"  Training samples  : {train_data.samples}")
print(f"  Validation samples: {val_data.samples}")
print(f"  Classes           : {train_data.class_indices}")

# Build model
print("\n  Building MobileNetV2 model...")
model, base_model = build_model()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()
print(f"\n  Total trainable parameters: {model.count_params():,}")

# Phase 1 callbacks — save weights only (version-independent)
callbacks_p1 = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-7, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath='mobilenetv2_weights.weights.h5',
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,       # ← weights only, no architecture
        verbose=0
    )
]

# Phase 1: Train with frozen base
print("\n  Phase 1: Training classification head (frozen base)...")
print("=" * 60)

model.fit(
    train_data,
    validation_data=val_data,
    epochs=50,
    callbacks=callbacks_p1,
    verbose=1
)

# Phase 2: Fine-tune top layers
print("\n  Phase 2: Fine-tuning top layers...")
print("=" * 60)

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

train_data = train_datagen.flow_from_directory(
    train_dir, target_size=(224, 224), batch_size=32, class_mode='binary'
)
val_data = val_datagen.flow_from_directory(
    val_dir, target_size=(224, 224), batch_size=32, class_mode='binary', shuffle=False
)

callbacks_p2 = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath='mobilenetv2_weights.weights.h5',
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,       # ← weights only, no architecture
        verbose=0
    )
]

model.fit(
    train_data,
    validation_data=val_data,
    epochs=50,
    callbacks=callbacks_p2,
    verbose=1
)

# ============================================================================
# STEP 6: EVALUATE BEST WEIGHTS
# ============================================================================

print("\n" + "=" * 60)
print("STEP 6: EVALUATING BEST MODEL")
print("=" * 60)

val_data = val_datagen.flow_from_directory(
    val_dir, target_size=(224, 224), batch_size=32, class_mode='binary', shuffle=False
)

# Reload best weights into a fresh model for clean evaluation
eval_model, _ = build_model()
eval_model.build((None, 224, 224, 3))
eval_model.load_weights('mobilenetv2_weights.weights.h5')
eval_model.compile(loss='binary_crossentropy', metrics=['accuracy'])

test_loss, test_accuracy = eval_model.evaluate(val_data, verbose=0)

weights_size = os.path.getsize('mobilenetv2_weights.weights.h5') / (1024 * 1024)
print(f"\n  Val Loss      : {test_loss:.4f}")
print(f"  Val Accuracy  : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"  Weights size  : {weights_size:.2f} MB")


# ============================================================================
# STEP 7: CREATE SUBMISSION ZIP
# ============================================================================

print("\n" + "="*60)
print("STEP 7: CREATING SUBMISSION PACKAGE")
print("="*60)

submission_files = [
    'test.py',
    'mobilenetv2_weights.weights.h5',
    'requirements.txt',
]

if os.path.exists('submission.zip'):
    os.remove('submission.zip')

print("\n  Adding files to submission package:")
with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in submission_files:
        if os.path.exists(file):
            zipf.write(file)
            file_size = os.path.getsize(file) / 1024
            print(f"    ✓ {file:40s} ({file_size:8.1f} KB)")
        else:
            print(f"    ✗ {file:40s} NOT FOUND — skipped")

zip_size = os.path.getsize('submission.zip') / (1024 * 1024)
print(f"\n  Package created: submission.zip ({zip_size:.2f} MB)")

print("\n  Verifying package contents:")
with zipfile.ZipFile('submission.zip', 'r') as zipf:
    for info in zipf.filelist:
        print(f"    ✓ {info.filename:40s} ({info.file_size / 1024:8.1f} KB)")


# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 60)
print("🎉 SUBMISSION PACKAGE READY!")
print("=" * 60)

print(f"\n  Main Package  : submission.zip ({zip_size:.2f} MB)")
print(f"  Val Accuracy  : {test_accuracy*100:.2f}%")

print("\n  Package Contents:")
print("    ✓ test.py                             - Main inference script")
print("    ✓ mobilenetv2_weights.weights.h5      - Trained weights (version-safe)")
print("    ✓ requirements.txt                    - Dependencies")

print("\n  Why weights-only?")
print("    Saving the full .keras model embeds Keras version metadata.")
print("    The grader's older Keras can't load 'quantization_config'.")
print("    Weights-only + rebuild avoids this entirely.")

print("\n  Quick Start:")
print("    1. Extract submission.zip")
print("    2. pip install -r requirements.txt")
print("    3. python test.py <your_image_directory>")

print("\n  ⬇️  Download submission.zip from the Colab file panel (left sidebar)")
print("=" * 60)

🐱🐶 CATS VS DOGS SUBMISSION CREATOR
Timestamp: 2026-03-20 16:18:04
TensorFlow version : 2.19.0
Keras version      : 3.13.2

📄 Creating test.py...
  ✓ test.py created

📄 Creating requirements.txt...
  ✓ requirements.txt created

STEP 5: TRAINING MobileNetV2 MODEL
This will take 15-20 minutes. Training in progress...

STEP 3: DOWNLOADING CATS VS DOGS DATASET
  Dataset already exists locally!

  Setting up data generators...
Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.
  Training samples  : 2000
  Validation samples: 1000
  Classes           : {'cats': 0, 'dogs': 1}

  Building MobileNetV2 model...


Model: "MobileNetV2_CatsDogs"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_6      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,618,945 (9.99 MB)

 Trainable params: 360,961 (1.38 MB)

 Non-trainable params: 2,257,984 (8.61 MB)


  Total trainable parameters: 2,618,945

  Phase 1: Training classification head (frozen base)...
Epoch 1/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 51s 653ms/step - accuracy: 0.7990 - loss: 0.4438 - val_accuracy: 0.9700 - val_loss: 0.1378 - learning_rate: 1.0000e-04
Epoch 2/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 27s 431ms/step - accuracy: 0.9280 - loss: 0.1969 - val_accuracy: 0.9780 - val_loss: 0.0772 - learning_rate: 1.0000e-04
Epoch 3/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 27s 437ms/step - accuracy: 0.9320 - loss: 0.1711 - val_accuracy: 0.9810 - val_loss: 0.0665 - learning_rate: 1.0000e-04
Epoch 4/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 27s 428ms/step - accuracy: 0.9435 - loss: 0.1421 - val_accuracy: 0.9790 - val_loss: 0.0636 - learning_rate: 1.0000e-04
Epoch 5/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 27s 431ms/step - accuracy: 0.9485 - loss: 0.1271 - val_accuracy: 0.9790 - val_loss: 0.0567 - learning_rate: 1.0000e-04
Epoch 6/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 30s 469ms/step - accuracy: 0.9455 - loss: 0.1355 - val_accuracy: 0.9790 - val_